# Stage 3 Ablation Study: FFT On vs FFT Off and Label Shuffle

This notebook evaluates the Stage 3 non-localisation classifier under three controlled settings:

- FFT ON: full RGB-frequency fusion model
- FFT OFF: RGB-only ablation
- Label Shuffle: sanity-check experiment with randomized training labels

This version uses a stricter FFT ablation and a memory-safe label shuffle implementation.


In [1]:
import os
import random
import warnings
from io import BytesIO

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision.transforms.functional as TF
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset, Subset

from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

# --------------------------------------------
# Config
# --------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/hooriyamasood80/140k-faces-aligned/kaggle/working/140k_aligned_faces"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
EPOCHS = 2
BATCH_SIZE = 16
LR = 1e-4

RUN_FFT_ON = True
RUN_FFT_OFF = True
RUN_LABEL_SHUFFLE = True

SMALL_RUN = False
N_PER_CLASS = 100


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# y_true must be 1=fake, 0=real
def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    return auc, f1, eer, acc

def print_results(tag, auc, f1, eer, acc):
    print(f"{tag} -> AUROC={auc:.6f} | F1={f1:.6f} | EER={eer:.6f} | ACC={acc:.6f}")


In [3]:
class RandomJPEG:
    def __init__(self, quality_min=30, quality_max=100, p=0.7):
        self.quality_min = quality_min
        self.quality_max = quality_max
        self.p = p

    def __call__(self, img: Image.Image):
        if random.random() > self.p:
            return img
        q = random.randint(self.quality_min, self.quality_max)
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

class DualViewImageFolder(datasets.ImageFolder):
    def __init__(self, root, p_flip=0.5, jpeg_aug=None, rgb_color_aug=None, rgb_norm=None):
        super().__init__(root, transform=None)
        self.p_flip = p_flip
        self.jpeg_aug = jpeg_aug
        self.rgb_color_aug = rgb_color_aug
        self.rgb_norm = rgb_norm

    def __getitem__(self, index):
        path, y = self.samples[index]
        img = self.loader(path).convert("RGB")

        if random.random() < self.p_flip:
            img = TF.hflip(img)

        img_jpeg = self.jpeg_aug(img) if self.jpeg_aug is not None else img

        x_raw = TF.to_tensor(img_jpeg)

        img_rgb = img_jpeg
        if self.rgb_color_aug is not None:
            img_rgb = self.rgb_color_aug(img_rgb)

        x_rgb = TF.to_tensor(img_rgb)
        if self.rgb_norm is not None:
            x_rgb = self.rgb_norm(x_rgb)

        return x_rgb, x_raw, y


In [4]:
def make_balanced_subset_from_targets(ds, n_per_class=100, seed=42):
    rng = np.random.RandomState(seed)
    targets = np.array(ds.targets)
    idxs = []

    for c in np.unique(targets):
        c_idx = np.where(targets == c)[0]
        take = min(n_per_class, len(c_idx))
        idxs.extend(rng.choice(c_idx, size=take, replace=False).tolist())

    rng.shuffle(idxs)
    return Subset(ds, idxs)

class ShuffledLabelDataset(Dataset):
    def __init__(self, base_ds, seed=42):
        self.base = base_ds
        rng = np.random.RandomState(seed)

        if hasattr(base_ds, "targets"):
            labels = np.array(base_ds.targets)
        elif hasattr(base_ds, "samples"):
            labels = np.array([y for _, y in base_ds.samples])
        else:
            raise ValueError("Base dataset must expose .targets or .samples")

        self.new_labels = rng.permutation(labels)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x_rgb, x_raw, _ = self.base[idx]
        return x_rgb, x_raw, int(self.new_labels[idx])


In [5]:
class GlobalFilter(nn.Module):
    def __init__(self, dim, h, w, fp32fft=True):
        super().__init__()
        self.h = h
        self.w = w
        self.fp32fft = fp32fft
        self.complex_weight = nn.Parameter(
            torch.randn(h, w // 2 + 1, dim, 2, dtype=torch.float32) * 0.02
        )

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.h and W == self.w, f"Expected {self.h}x{self.w}, got {H}x{W}"

        x = x.permute(0, 2, 3, 1).contiguous()

        if self.fp32fft:
            orig_dtype = x.dtype
            x = x.float()

        x_f = torch.fft.rfft2(x, dim=(1, 2), norm="ortho")
        w = torch.view_as_complex(self.complex_weight)
        x_f = x_f * w
        x = torch.fft.irfft2(x_f, s=(H, W), dim=(1, 2), norm="ortho")

        if self.fp32fft:
            x = x.to(orig_dtype)

        x = x.permute(0, 3, 1, 2).contiguous()
        return x


In [6]:
class Stage3Hybrid(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, use_fft=True, grid=10):
        super().__init__()
        self.use_fft = use_fft
        self.grid = grid
        self.freq_filter = GlobalFilter(dim=512, h=28, w=28, fp32fft=True)

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.proj_rgb = nn.Conv2d(2048, embed_dim, 1)
        self.proj_fft = nn.Conv2d(512, embed_dim, 1)

        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.type_embed = nn.Parameter(torch.randn(1, 2, embed_dim))

        num_tokens = 1 + 2 * (grid * grid)
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens, embed_dim))

        self.cls_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2)
        )

    def forward(self, x_rgb, x_raw=None):
        B = x_rgb.size(0)

        x = self.stem(x_rgb)
        x = self.layer1(x)
        feat_l2 = self.layer2(x)

        x = self.layer3(feat_l2)
        feat_l4 = self.layer4(x)

        rgb_map = self.proj_rgb(feat_l4)
        rgb_map = F.adaptive_avg_pool2d(rgb_map, (self.grid, self.grid))
        rgb_tok = rgb_map.flatten(2).transpose(1, 2)
        rgb_tok = rgb_tok + self.type_embed[:, 0:1, :]

        cls = self.cls_token.repeat(B, 1, 1)

        if self.use_fft:
            freq_feat = self.freq_filter(feat_l2)
            fft_map = self.proj_fft(freq_feat)
            fft_map = F.adaptive_avg_pool2d(fft_map, (self.grid, self.grid))
            fft_tok = fft_map.flatten(2).transpose(1, 2)
            fft_tok = fft_tok + self.type_embed[:, 1:2, :]

            fused_rgb, _ = self.cross_attn(query=rgb_tok, key=fft_tok, value=fft_tok)
            rgb_tok = rgb_tok + fused_rgb

            tokens = torch.cat([cls, rgb_tok, fft_tok], dim=1)
        else:
            tokens = torch.cat([cls, rgb_tok], dim=1)

        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]

        out = self.transformer(tokens)
        cls_out = out[:, 0]
        return self.cls_head(cls_out)


In [7]:
def evaluate(model, loader, fake_idx, tag="TEST"):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for x_rgb, x_raw, lbls in tqdm(loader, desc=tag, ncols=100):
            x_rgb = x_rgb.to(DEVICE, non_blocking=True)
            x_raw = x_raw.to(DEVICE, non_blocking=True)

            logits = model(x_rgb, x_raw)
            probs_fake = torch.softmax(logits, dim=1)[:, fake_idx].cpu().numpy()

            lbls_np = lbls.numpy()
            y_true.extend((lbls_np == fake_idx).astype(np.int32))
            y_prob.extend(probs_fake)

    auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
    print_results(tag, auc, f1, eer, acc)
    return auc, f1, eer, acc


In [8]:
def train_stage3(
    data_root,
    epochs=10,
    batch_size=16,
    lr=1e-4,
    small_run=False,
    n_per_class=100,
    shuffle_labels=False,
    no_fft=False,
    seed=42,
    ckpt_name="best_auc_stage3.pth",
):
    set_seed(seed)

    model = Stage3Hybrid(use_fft=(not no_fft)).to(DEVICE)
    print("use_fft =", model.use_fft)

    jpeg_aug = RandomJPEG(quality_min=30, quality_max=100, p=0.7)
    rgb_color_aug = transforms.Compose([
        transforms.ColorJitter(0.2, 0.2, 0.1, 0.05),
    ])
    rgb_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    trainset = DualViewImageFolder(
        os.path.join(data_root, "train"),
        p_flip=0.5,
        jpeg_aug=jpeg_aug,
        rgb_color_aug=rgb_color_aug,
        rgb_norm=rgb_norm,
    )
    valset = DualViewImageFolder(
        os.path.join(data_root, "valid"),
        p_flip=0.0,
        jpeg_aug=None,
        rgb_color_aug=None,
        rgb_norm=rgb_norm,
    )
    testset = DualViewImageFolder(
        os.path.join(data_root, "test"),
        p_flip=0.0,
        jpeg_aug=None,
        rgb_color_aug=None,
        rgb_norm=rgb_norm,
    )

    if small_run:
        print(f"SMALL RUN: {n_per_class} per class")
        trainset = make_balanced_subset_from_targets(trainset, n_per_class=n_per_class, seed=seed)
        valset = make_balanced_subset_from_targets(valset, n_per_class=n_per_class, seed=seed + 1)
        testset = make_balanced_subset_from_targets(testset, n_per_class=n_per_class, seed=seed + 2)

    class_to_idx = trainset.dataset.class_to_idx if isinstance(trainset, Subset) else trainset.class_to_idx

    if shuffle_labels:
        print("LABEL SHUFFLE ON (train labels randomized)")
        trainset = ShuffledLabelDataset(trainset, seed=seed)

    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    fake_idx = class_to_idx["fake"]
    print("class_to_idx:", class_to_idx)

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())
    best_auc = -1.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x_rgb, x_raw, lbls in tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}", ncols=100):
            x_rgb = x_rgb.to(DEVICE, non_blocking=True)
            x_raw = x_raw.to(DEVICE, non_blocking=True)
            lbls = lbls.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                out = model(x_rgb, x_raw)
                loss = criterion(out, lbls)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            running_loss += loss.item()

        auc, f1, eer, acc = evaluate(model, valloader, fake_idx, tag=f"VALID E{epoch+1}")
        print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f}")

        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), f"/kaggle/working/{ckpt_name}")
            print(f"New best AUC {best_auc:.6f} (ACC={acc:.6f})")

    print("\nTraining complete!")
    print(f"Best model achieved: AUROC={best_auc:.6f}")
    return model, testloader, fake_idx


In [9]:
if RUN_FFT_ON:
    print("\n==============================")
    print("RUN A: FFT ON")
    print("==============================")

    model_on, testloader_on, fake_idx_on = train_stage3(
        data_root=DATA_ROOT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        small_run=SMALL_RUN,
        n_per_class=N_PER_CLASS,
        shuffle_labels=False,
        no_fft=False,
        seed=SEED,
        ckpt_name="best_auc_stage3_fft_on.pth",
    )

    model_on.load_state_dict(torch.load("/kaggle/working/best_auc_stage3_fft_on.pth", map_location=DEVICE))
    model_on.eval()

    fft_on_auc, fft_on_f1, fft_on_eer, fft_on_acc = evaluate(
        model_on, testloader_on, fake_idx_on, tag="TEST FFT ON"
    )



RUN A: FFT ON
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]


use_fft = True
class_to_idx: {'fake': 0, 'real': 1}


VALID E1: 100%|█████████████████████████████████████████████████| 1250/1250 [01:53<00:00, 11.00it/s]


VALID E1 -> AUROC=0.996966 | F1=0.947954 | EER=0.027600 | ACC=0.950400
Epoch 1: loss=0.1900
New best AUC 0.996966 (ACC=0.950400)


VALID E2: 100%|█████████████████████████████████████████████████| 1250/1250 [01:53<00:00, 11.06it/s]


VALID E2 -> AUROC=0.998881 | F1=0.984098 | EER=0.015800 | ACC=0.984200
Epoch 2: loss=0.0884
New best AUC 0.998881 (ACC=0.984200)

Training complete!
Best model achieved: AUROC=0.998881


TEST FFT ON: 100%|██████████████████████████████████████████████| 1250/1250 [01:56<00:00, 10.72it/s]

TEST FFT ON -> AUROC=0.999107 | F1=0.985974 | EER=0.013400 | ACC=0.986050


In [10]:
if RUN_FFT_OFF:
    print("\n==============================")
    print("RUN B: FFT OFF (RGB ONLY)")
    print("==============================")

    model_off, testloader_off, fake_idx_off = train_stage3(
        data_root=DATA_ROOT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        small_run=SMALL_RUN,
        n_per_class=N_PER_CLASS,
        shuffle_labels=False,
        no_fft=True,
        seed=SEED,
        ckpt_name="best_auc_stage3_fft_off.pth",
    )

    model_off.load_state_dict(torch.load("/kaggle/working/best_auc_stage3_fft_off.pth", map_location=DEVICE))
    model_off.eval()

    fft_off_auc, fft_off_f1, fft_off_eer, fft_off_acc = evaluate(
        model_off, testloader_off, fake_idx_off, tag="TEST FFT OFF"
    )



RUN B: FFT OFF (RGB ONLY)
use_fft = False
class_to_idx: {'fake': 0, 'real': 1}


VALID E1: 100%|█████████████████████████████████████████████████| 1250/1250 [01:22<00:00, 15.07it/s]


VALID E1 -> AUROC=0.998324 | F1=0.950003 | EER=0.019600 | ACC=0.952250
Epoch 1: loss=0.1845
New best AUC 0.998324 (ACC=0.952250)


VALID E2: 100%|█████████████████████████████████████████████████| 1250/1250 [01:22<00:00, 15.06it/s]


VALID E2 -> AUROC=0.998755 | F1=0.977410 | EER=0.016200 | ACC=0.977050
Epoch 2: loss=0.0877
New best AUC 0.998755 (ACC=0.977050)

Training complete!
Best model achieved: AUROC=0.998755


TEST FFT OFF: 100%|█████████████████████████████████████████████| 1250/1250 [01:22<00:00, 15.08it/s]

TEST FFT OFF -> AUROC=0.998904 | F1=0.979453 | EER=0.014500 | ACC=0.979150


In [11]:
if RUN_LABEL_SHUFFLE:
    print("\n==============================")
    print("RUN C: LABEL SHUFFLE")
    print("==============================")

    model_shuffle, testloader_shuffle, fake_idx_shuffle = train_stage3(
        data_root=DATA_ROOT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        small_run=SMALL_RUN,
        n_per_class=N_PER_CLASS,
        shuffle_labels=True,
        no_fft=False,
        seed=SEED,
        ckpt_name="best_auc_stage3_label_shuffle.pth",
    )

    model_shuffle.load_state_dict(torch.load("/kaggle/working/best_auc_stage3_label_shuffle.pth", map_location=DEVICE))
    model_shuffle.eval()

    shuffle_auc, shuffle_f1, shuffle_eer, shuffle_acc = evaluate(
        model_shuffle, testloader_shuffle, fake_idx_shuffle, tag="TEST LABEL SHUFFLE"
    )



RUN C: LABEL SHUFFLE
use_fft = True
LABEL SHUFFLE ON (train labels randomized)
class_to_idx: {'fake': 0, 'real': 1}


VALID E1: 100%|█████████████████████████████████████████████████| 1250/1250 [01:53<00:00, 11.00it/s]


VALID E1 -> AUROC=0.450808 | F1=0.666667 | EER=0.543662 | ACC=0.500000
Epoch 1: loss=0.6985
New best AUC 0.450808 (ACC=0.500000)


VALID E2: 100%|█████████████████████████████████████████████████| 1250/1250 [01:52<00:00, 11.11it/s]


VALID E2 -> AUROC=0.503749 | F1=0.000000 | EER=0.492443 | ACC=0.500000
Epoch 2: loss=0.6944
New best AUC 0.503749 (ACC=0.500000)

Training complete!
Best model achieved: AUROC=0.503749


TEST LABEL SHUFFLE: 100%|███████████████████████████████████████| 1250/1250 [01:52<00:00, 11.14it/s]

TEST LABEL SHUFFLE -> AUROC=0.514533 | F1=0.000000 | EER=0.484211 | ACC=0.500000


In [12]:
print("\nFinal Ablation Results")
print("-" * 60)

if RUN_FFT_ON:
    print(f"FFT ON        -> AUROC={fft_on_auc:.6f} | F1={fft_on_f1:.6f} | EER={fft_on_eer:.6f} | ACC={fft_on_acc:.6f}")

if RUN_FFT_OFF:
    print(f"FFT OFF       -> AUROC={fft_off_auc:.6f} | F1={fft_off_f1:.6f} | EER={fft_off_eer:.6f} | ACC={fft_off_acc:.6f}")

if RUN_LABEL_SHUFFLE:
    print(f"LABEL SHUFFLE -> AUROC={shuffle_auc:.6f} | F1={shuffle_f1:.6f} | EER={shuffle_eer:.6f} | ACC={shuffle_acc:.6f}")



Final Ablation Results
------------------------------------------------------------
FFT ON        -> AUROC=0.999107 | F1=0.985974 | EER=0.013400 | ACC=0.986050
FFT OFF       -> AUROC=0.998904 | F1=0.979453 | EER=0.014500 | ACC=0.979150
LABEL SHUFFLE -> AUROC=0.514533 | F1=0.000000 | EER=0.484211 | ACC=0.500000
